# 05 — Evaluation & LogisChain Lab Simulation
Model evaluation summary (Section A6) and the LogisChain Lab gamified simulation (Part B), comparing an SC-aware policy against a passive, financial-only baseline.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import json
from pathlib import Path
results_path = Path('..') / 'results' / 'pipeline_results.json'
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(json.dumps(results, indent=2)[:2000])
else:
    print('Run `python -m demo.run_pipeline` first to generate results/pipeline_results.json')

{
  "data": {
    "n_nodes": 217,
    "n_edges": 1223,
    "n_transportation_links": 634,
    "n_shipments": 3000,
    "n_lc_transactions": 3000,
    "default_rate_nodes": 0.07373271889400922,
    "default_rate_lc": 0.06366666666666666
  },
  "gnn": {
    "node_classification_accuracy": 0.7727272727272727,
    "supplier_risk_tier_accuracy": 0.8333333333333334,
    "link_prediction_auc": 0.637850667734279
  },
  "tcn": {
    "throughput": {
      "mape_30d_by_entity": {
        "Los Angeles": 2.0285702372408685,
        "Rotterdam": 2.212475932772288,
        "Shanghai": 12.57685244078654,
        "Singapore": 22.386844201874435
      },
      "mape_30d_mean": 9.801185703168532
    },
    "freight_rate": {
      "mape_30d_by_entity": {
        "Los Angeles": 12.056915350753759,
        "Rotterdam": 12.637415440695518,
        "Shanghai": 19.060333191781634,
        "Singapore": 21.21927233268806
      },
      "mape_30d_mean": 16.243484078979744
    }
  },
  "transformer": {
    "delay_

## LogisChain Lab: Trade Finance Portfolio Management + SCF Pricing
Both mandatory game modes (Section B1.3), run under both policies over an identical scenario sequence.

In [2]:
from src.data.synthetic_generator import SupplyChainDataGenerator
from src.simulation.game_modes import TradeFinancePortfolioMode, SupplyChainFinancePricingMode

gen = SupplyChainDataGenerator(seed=42)
nodes, edges = gen.generate_graph()

mode1 = TradeFinancePortfolioMode(nodes, edges, n_clients=45, seed=7)
res_sc = mode1.run('sc_aware', n_turns=52)
res_passive = mode1.run('passive', n_turns=52)
print('SC-aware score:', res_sc['score']['total'], res_sc['score']['certification_level'])
print('Passive score:', res_passive['score']['total'], res_passive['score']['certification_level'])

SC-aware score: 795.6 Expert
Passive score: 421.5 Practitioner


In [3]:
mode2 = SupplyChainFinancePricingMode(nodes, edges, n_suppliers=60, seed=7)
res2_sc = mode2.run('sc_aware', n_turns=52)
res2_passive = mode2.run('passive', n_turns=52)
print('SC-aware revenue: $%.0f, score: %.1f' % (res2_sc['total_discount_revenue_usd'], res2_sc['score']['total']))
print('Passive revenue:  $%.0f, score: %.1f' % (res2_passive['total_discount_revenue_usd'], res2_passive['score']['total']))

SC-aware revenue: $538728, score: 730.1
Passive revenue:  $76923, score: 374.4


## Scenario log — which disruptions were triggered this run

In [4]:
import pandas as pd
pd.DataFrame(res_sc['scenario_log'])

,turn,scenario,duration_turns,ports
0,7,Demand Whiplash (Bullwhip),5,"[PORT_0003, PORT_0011]"
1,18,Geopolitical Trade Route Closure,7,"[PORT_0000, PORT_0009]"
2,22,Port Congestion Event,1,"[PORT_0005, PORT_0000]"
3,30,Regulatory/Sanctions Change,16,"[PORT_0005, PORT_0007]"
4,31,Natural Disaster (Typhoon/Flood),3,"[PORT_0004, PORT_0011]"
5,40,Pandemic-Style Disruption,31,"[PORT_0010, PORT_0005]"
